# Zebrafish embryogenesis

This notebook runs the packaged `zebrafish` workflow from data preparation
through downstream analysis. Edit the paths in **Setup**, then enable the run
switches for the steps you need. The saved outputs below come from the packaged
preset and do not require the external dataset.

## Setup

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

from CytoBridge.workflow import (
    WorkflowOptions,
    build_workflow_plan,
    load_workflow_config,
    render_workflow_plan,
    run_workflow,
)
PRESET = 'zebrafish'
RAW_H5AD = Path("data/zebrafish_raw.h5ad")
OUTPUT_DIR = Path("tutorial_outputs/zebrafish")
ALIGNED_H5AD = OUTPUT_DIR / "preprocess" / 'zebrafish_aligned.h5ad'
MODEL_DIR = OUTPUT_DIR / "training"


RUN_PREPARATION = False
RUN_PREPROCESS_AND_TRAIN = False
RUN_DOWNSTREAM = False

In [2]:
config, preset_source = load_workflow_config(PRESET)
dataset = config["dataset"]
scientific = config["scientific"]
downstream = config["downstream"]

pd.DataFrame(
    {
        "setting": [
            "dataset",
            "preset",
            "raw time column",
            "cell annotation",
            "observed training times",
            "classifier neighbors",
        ],
        "value": [
            dataset["display_name"],
            preset_source,
            config["preprocess"]["time_key"],
            dataset["annotation_key"],
            ", ".join(map(str, downstream["observed"])),
            scientific["classifier_k"],
        ],
    }
)

,setting,value
0,dataset,Zebrafish embryogenesis
1,preset,packaged preset: zebrafish
2,raw time column,time
3,cell annotation,Annotation
4,observed training times,"0.0, 1.0, 2.0, 3.0, 4.0"
5,classifier neighbors,10


## Files passed from one step to the next

These are the handoffs used below. Training reads the aligned H5AD and edge
predictor written by preprocessing. Downstream analysis then reads that same
aligned H5AD together with the complete training directory.

### Step 1: preprocess

```text
cytobridge workflow --config zebrafish --step preprocess --input-h5ad <raw.h5ad> --output-dir <run>
```

**Reads:** `raw H5AD and packaged dataset preset`

**Writes:** `<run>/preprocess/zebrafish_aligned.h5ad; <run>/preprocess/edge_classifier/zebrafish_edge_model.pt; preprocessing manifests`

**Next:** `training`

**Availability:** public

### Step 2: preprocess and train

```text
cytobridge workflow --config zebrafish --step preprocess --step train --train --input-h5ad <raw.h5ad> --output-dir <run> --device cuda
```

**Reads:** `raw H5AD; packaged dataset preset; packaged LR database`

**Writes:** `<run>/training/<stage>/best_model.pth or score_model.pth; <run>/training/adata.h5ad; training_history.csv; training_run_summary.json`

**Next:** `downstream`

**Availability:** public

### Step 3: downstream

```text
cytobridge workflow --config zebrafish --step downstream --aligned-h5ad <run>/preprocess/zebrafish_aligned.h5ad --model-dir <run>/training --output-dir <run>
```

**Reads:** `aligned H5AD; <run>/training; dataset-matched LR database`

**Writes:** `<run>/downstream/summary.json; slice_data/*.h5ad; velocity/velocity_components.npz; growth/growth_by_cell.csv; composition/celltype_composition.csv; communication and ligand_receptor tables; standard figures`

**Next:** `paper-specific continuation shown in the paper-figure notebook`

**Availability:** public

## Data preparation

The preset records the count layer, time mapping, spatial coordinates, and
alignment settings used for this dataset. The plan below shows the input and
output paths before any long-running work starts.

In [3]:
preparation_options = WorkflowOptions(
    input_h5ad=RAW_H5AD,
    output_dir=OUTPUT_DIR,
    steps=("preprocess",),
)
preparation_plan = build_workflow_plan(
    config,
    source=preset_source,
    options=preparation_options,
)
print(render_workflow_plan(preparation_plan))

CytoBridge workflow plan
dataset: Zebrafish embryogenesis (zebrafish)
config: packaged preset: zebrafish
scientific parameters: alpha_spatial=10, alpha_express=0.015, seed=42, classifier_k=10
steps:
  preprocess: ready (GPU recommended for spatial alignment)
    output: tutorial_outputs/zebrafish/preprocess/zebrafish_aligned.h5ad
    edge predictor: not requested during preprocessing
  train: skipped; add --train to run (GPU required for production training)
  downstream: skipped (GPU recommended)


In [4]:
if RUN_PREPARATION:
    if not RAW_H5AD.is_file():
        raise FileNotFoundError(f"Update RAW_H5AD before preprocessing: {RAW_H5AD}")
    preparation_result = run_workflow(config, options=preparation_options)
    preparation_result
else:
    print("Data preparation is off. Set RUN_PREPARATION = True to run it.")

Data preparation is off. Set RUN_PREPARATION = True to run it.


## Training

The full training run starts from the raw H5AD, writes the aligned data, fits
the interaction edge predictor when the preset requires one, and trains the
CytoBridge model. A production run requires a CUDA-capable environment.

In [5]:
training_options = WorkflowOptions(
    input_h5ad=RAW_H5AD,
    output_dir=OUTPUT_DIR,
    steps=("preprocess", "train"),
    train=True,
)
training_plan = build_workflow_plan(
    config,
    source=preset_source,
    options=training_options,
)
print(render_workflow_plan(training_plan))

CytoBridge workflow plan
dataset: Zebrafish embryogenesis (zebrafish)
config: packaged preset: zebrafish
scientific parameters: alpha_spatial=10, alpha_express=0.015, seed=42, classifier_k=10
steps:
  preprocess: ready (GPU recommended for spatial alignment)
    output: tutorial_outputs/zebrafish/preprocess/zebrafish_aligned.h5ad
    edge predictor: will be trained automatically
      graph database: package: CytoBridge/workflow_databases/CellChatDB.ligrec.zebrafish.csv
      database source: bundled formal CellChatDB resource
      interaction cutoff: 0.09606367405591873
      decision threshold source: validation-selected during de novo training
      output: tutorial_outputs/zebrafish/preprocess/edge_classifier/zebrafish_edge_model.pt
  train: ready (GPU required for production training)
    training config: zebrafish_spatial_full_alpha_express_0015.yaml
    interaction cutoff: 0.09606367405591873
    edge predictor threshold source: validation-selected during preprocessing
    edge

In [6]:
if RUN_PREPROCESS_AND_TRAIN:
    if not RAW_H5AD.is_file():
        raise FileNotFoundError(f"Update RAW_H5AD before training: {RAW_H5AD}")
    training_result = run_workflow(config, options=training_options)
    training_result
else:
    print("Training is off. Set RUN_PREPROCESS_AND_TRAIN = True to start it.")

Training is off. Set RUN_PREPROCESS_AND_TRAIN = True to start it.


## Downstream analysis

Downstream analysis uses the aligned H5AD and fitted model from the training
directory. The dataset preset supplies the interpolation times, classifier
settings, trajectory simulation, growth analysis, and ligand–receptor options.

In [7]:
downstream_options = WorkflowOptions(
    aligned_h5ad=ALIGNED_H5AD,
    model_dir=MODEL_DIR,
    output_dir=OUTPUT_DIR,
    steps=("downstream",),
)
downstream_plan = build_workflow_plan(
    config,
    source=preset_source,
    options=downstream_options,
)
print(render_workflow_plan(downstream_plan))

CytoBridge workflow plan
dataset: Zebrafish embryogenesis (zebrafish)
config: packaged preset: zebrafish
scientific parameters: alpha_spatial=10, alpha_express=0.015, seed=42, classifier_k=10
steps:
  preprocess: skipped (GPU for spatial alignment)
  train: skipped; add --train to run (GPU required for production training)
  downstream: ready (GPU recommended for SDE simulation and classifier fitting)
    model format: current
    output: tutorial_outputs/zebrafish/downstream
    simulation: observed=[0.0, 1.0, 2.0, 3.0, 4.0], interpolated=[0.5, 1.5, 2.5, 3.5], initial particles=all observed t0 cells, dt=0.05, sigma=0.03, daughter noise=0, growth alpha=1, trajectory mode=piecewise_observed_anchored_interval_forward_simulation
      piecewise observed anchors: sample mode=per_timepoint, include end=False
      scope: Observed times use real cells. Each generated time is a one-sided, interval-local forward simulation initialized only from the immediately preceding observed anchor; it is 

In [8]:
if RUN_DOWNSTREAM:
    missing = [path for path in (ALIGNED_H5AD, MODEL_DIR) if not path.exists()]
    if missing:
        raise FileNotFoundError(f"Missing trained artifacts: {missing}")
    downstream_result = run_workflow(config, options=downstream_options)
    downstream_result
else:
    print("Downstream analysis is off. Set RUN_DOWNSTREAM = True to run it.")

Downstream analysis is off. Set RUN_DOWNSTREAM = True to run it.


## Paper figures

The steps below show exactly where this dataset's standard downstream output
continues into manuscript calculations. A step marked `provenance break` means
that a related calculation exists but the manuscript page cannot yet be traced
to one exact command and input set.

- [Supplementary Figures S31–S38](../paper_figures/zebrafish_si_s31_s38.ipynb)
- [Supplementary Figure S43](../paper_figures/zebrafish_attention.ipynb)

### Step 1: calculate the manuscript downstream bundle

**Paper:** S31-S35; S38

```text
python -m scripts.run_zebrafish_paper_downstream --aligned-h5ad <run>/preprocess/zebrafish_aligned.h5ad --model-dir <run>/training --lr-database <zebrafish-lr.csv> --output-dir <paper-run> --stage all --profile smoke --device cuda
```

**Reads:** `aligned H5AD; six-stage model; manuscript run record; zebrafish LR database`

**Writes:** `global-t0 state transport; growth; virtual-removal; gene-dynamics; inverse-PCA; communication tables and manifests`

**Next:** `run the S31-S38 paper notebook with a schema-matched compact bundle`

**Availability:** public smoke command; the full manuscript profile additionally reads its matching run record

### Step 2: run the two sensitivity analyses

**Paper:** S36-S37

```text
run the matched loss-weight sweep; inspect the fixed daughter-noise runner with `python -m scripts.run_zebrafish_interval_daughter_noise_sensitivity --help`
```

**Reads:** `matched zebrafish checkpoints and fixed evaluation cohorts`

**Writes:** `loss-weight and daughter-noise CSV tables with run manifests`

**Next:** `run the S31-S38 paper notebook`

**Availability:** loss-weight runner retained with the manuscript run; daughter-noise runner public

### Step 3: calculate and render attention validation

**Paper:** S43

```text
python -m scripts.run_zebrafish_attention_validation analyze --spec <analysis-spec.json> --output-dir <attention-analysis> --n-selected-pairs 30; inspect the report inputs with `python -m scripts.run_zebrafish_attention_validation report --help`
```

**Reads:** `manuscript model attention; aligned cells; fixed LR universe; COMMOT and CellAgentChat results`

**Writes:** `frozen validation tables, report manifest, vector PDF and PNG`

**Next:** `validate with the same script, then use the S43 notebook for the compact redraw`

**Availability:** public

## Saved files

- Aligned data: `tutorial_outputs/zebrafish/preprocess/zebrafish_aligned.h5ad`
- Training directory: `tutorial_outputs/zebrafish/training`
- Downstream directory: `tutorial_outputs/zebrafish/downstream`